## 1. Basic settings

In [ ]:
import os
from pathlib import Path
import random
import logging
import torch
import numpy as np
import torch.optim as optim

from codes.get_normal_attribute import get_normal_attribute
from codes.load_attribute import load_attribute
from codes.pre_data import pre_data
from codes.epsilon_conditional import EpsModel_nowcast
from codes.diffusion import Diffusion
from codes.train import train_conditional


In [ ]:
n_gpus = 1
device = torch.device("cuda:0" if (torch.cuda.is_available() and n_gpus > 0) else "cpu")
print(device)

if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

seed = 42 
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)


## 2. DataLoader for training

In [ ]:
camels_path = Path('data_example/CAMELS')

basin_list = np.loadtxt(camels_path / 'basin_list.txt', dtype=str, encoding='utf-8')
np.random.shuffle(basin_list)  # Shuffle in-place

path_attribute = camels_path / 'camels_attributes'
normal_attribute = get_normal_attribute(basin_list, path_attribute)

time_step = 365
batch_size = 512 # or 256

forcing_sources = {'daymet': ['prcp(mm/day)', 'srad(W/m2)', 'tmax(C)', 'tmin(C)', 'vp(Pa)'],
                    'nldas':  ['prcp(mm/day)', 'srad(W/m2)', 'tmax(C)', 'tmin(C)', 'vp(Pa)'],
                    'maurer': ['prcp(mm/day)', 'srad(W/m2)', 'tmax(C)', 'tmin(C)', 'vp(Pa)']}


In [ ]:
# Note on date range selection:
# Recommended workflow:
# 1. First train with 1980-10-01 to 1990-09-30, validate on 1990-10-01 to 1995-09-30
#    to find the optimal guidance weight for each basin.
# 2. Then retrain with 1980-10-01 to 1995-09-30 to maximize training data,
#    and use the previously determined guidance weights for classifier-free guidance sampling.
#
# Tips for guidance weight selection:
# - Based on our experience, searching over [-0.2, -0.1, 0, 0.1, 0.2] is sufficient.
# - Use DDIM with reduced diffusion steps and fewer sampling rounds (20-30) to speed up the search process.
date_ranges = {
    "train_start": "1980-10-01T00:00:00",
    "train_end": "1990-09-30T00:00:00"
}


dataloader_list, std = pre_data(time_step, basin_list, is_train=True, is_valid=False, date_ranges=date_ranges,
                                 batch_size=batch_size, camels_path=camels_path, forcing_sources=forcing_sources)


## 3. Initialization of diffusion model

In [ ]:
eps_model = EpsModel_nowcast(input_size=15, 
                             static_attr_len=27,
                             hidden_size=256,
                             emb_dim=128
                             ).to(device)

n_steps = 1200  # The number of steps for the diffusion process

diffusion = Diffusion(eps_model=eps_model, n_steps=n_steps, device=device)


## 4. Model traing

In [ ]:
save_dir = 'epsmodel/conditional'
os.makedirs(save_dir, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[logging.FileHandler("log_conditional.log"), logging.StreamHandler()])

logger = logging.getLogger(__name__)


In [ ]:
epochs = 400

lr_schedule = [(50, 5e-5), (100, 1e-5), (200, 5e-6), (300, 1e-6)]

initial_lr = 1e-4
optimizer = optim.AdamW(eps_model.parameters(), lr=initial_lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)

static_attributes = [torch.tensor(load_attribute(basin, normal_attribute), dtype=torch.float32).unsqueeze(0).expand((batch_size, -1)).to(device) for basin in basin_list]


In [ ]:
assert len(dataloader_list) == len(std) == len(static_attributes), \
    f"Length mismatch: dataloaders={len(dataloader_list)}, stds={len(std)}, attributes={len(static_attributes)}"

In [ ]:
train_loss = train_conditional(dataloader_list, save_dir, eps_model, diffusion, optimizer, static_attributes, 
                              std, epochs, lr_schedule, logger, device)